# Gemma 4 × VESSL Cloud Domain Fine-Tuning

This notebook fine-tunes **Gemma 4 E4B** with LoRA on a small (~36-sample)
VESSL Cloud domain QA dataset, demonstrating domain adaptation on a single
A100 SXM (80 GB).

### What you'll do
1. Load Gemma 4 E4B from the Unsloth mirror (4-bit).
2. Attach a LoRA adapter (r=32).
3. Train for a few hundred steps on the bundled VESSL QA dataset.
4. Inspect generation before/after fine-tuning.
5. Save the adapter to `/shared` (Object storage) or `/tmp`.

### Prerequisites
- A VESSL Cloud workspace with an A100 SXM 80 GB GPU attached.
- Image with PyTorch + CUDA (e.g., `pytorch/pytorch:2.5.1-cuda12.4-cudnn9-devel`).
- Cluster storage mounted at `/root` (fast local cache) and Object storage
  at `/shared` (persistent outputs).

### Expected cost
About **$0.43** for a full run at $1.55/hr (as of 2026-04-16), ~16 min.

> ⚠️ **Known limitation:** 36 samples teach the *shape* of a VESSL-style
> answer, not specific factual claims. The resulting model will fabricate
> specific numbers (e.g., GPU prices). See `../benchmarks.md` for details.


---
## 1. Install dependencies

Skip this cell if you pre-installed everything via `requirements.txt`.


In [ ]:
# Install Unsloth + TRL + transformers + datasets.
# Skip if they're already installed in the workspace.
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth>=2024.9", "trl>=0.11", "transformers>=4.45", "datasets>=3.0",
])
print("✅ Dependencies installed.")


---
## 2. Load Gemma 4 E4B

We pull the 4-bit Unsloth mirror of Gemma 4 E4B. No Hugging Face token
required for the mirror, which keeps the notebook self-contained.


In [ ]:
import os, time, json
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    dtype=None,
    max_seq_length=2048,
    load_in_4bit=True,
    full_finetuning=False,
)
print(f"✅ Loaded unsloth/gemma-4-E4B-it in 4-bit.")


---
## 3. Attach LoRA adapter

We use `r=32, alpha=32, dropout=0` — larger than the usual `r=8` for
general instruction tuning, because we're teaching domain-specific
content with only ~36 samples. Larger LoRA capacity helps the model
memorise the VESSL-specific vocabulary.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA attached. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")


---
## 4. Load the VESSL Cloud QA dataset

The dataset (36 Q&A pairs, CC-BY-4.0) is bundled in this repo at
`../data/vessl-cloud-qa-dataset.json`. See `../data/DATASET_CARD.md`
for provenance and licensing.


In [ ]:
import json
import os

# Prefer the bundled dataset (when running from the cloned repo); fall
# back to common upload paths so the notebook also works if users upload
# the JSON manually in JupyterLab.
possible_paths = [
    "../data/vessl-cloud-qa-dataset.json",
    "vessl-cloud-qa-dataset.json",
    "/root/vessl-cloud-qa-dataset.json",
    os.path.expanduser("~/vessl-cloud-qa-dataset.json"),
]

dataset_path = next((p for p in possible_paths if os.path.exists(p)), None)

if dataset_path is None:
    raise FileNotFoundError(
        "vessl-cloud-qa-dataset.json not found. If you didn't clone the "
        "cookbook repo, upload the file to this notebook's folder first."
    )

with open(dataset_path, "r") as f:
    qa_data = json.load(f)

print(f"✅ Loaded {dataset_path}: {len(qa_data)} Q&A pairs")
print("\nFirst sample:")
print(f"  Q: {qa_data[0]['conversations'][0]['value'][:80]}...")
print(f"  A: {qa_data[0]['conversations'][1]['value'][:80]}...")


---
## 5. Format the dataset

Convert the `from/value` conversation format to Gemma 4's chat template.


In [ ]:
from datasets import Dataset

# Convert the JSON records to a HuggingFace Dataset.
vessl_dataset = Dataset.from_list(qa_data)

def format_vessl_qa(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        # Map {from, value} → {role, content} for Gemma 4's chat template.
        messages = []
        for turn in convo:
            role = "user" if turn["from"] == "human" else "model"
            messages.append({
                "role": role,
                "content": [{"type": "text", "text": turn["value"]}]
            })
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        texts.append(text)
    return {"text": texts}

vessl_dataset = vessl_dataset.map(format_vessl_qa, batched=True)
print(f"Formatted {len(vessl_dataset)} samples.")
print("\nFirst 300 chars of the formatted prompt:")
print(vessl_dataset[0]["text"][:300])


---
## 6. Train

36 samples × 3 epochs ≈ 54 effective steps. Learning rate is kept low
(1e-4) to preserve the model's base knowledge while injecting domain
facts. For the full 20-epoch run used in `../benchmarks.md`, see the
vesslctl batch-job path (`../batch-job/`).


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

SAVE_DIR = "/shared/gemma4-vessl-expert" if os.path.isdir("/shared") else "/tmp/gemma4-vessl-expert"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Starting VESSL domain fine-tuning...")
train_start = time.time()

vessl_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=vessl_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps=3,
        num_train_epochs=3,           # 3 epochs — small dataset needs repetition.
        learning_rate=1e-4,           # Low LR preserves base knowledge.
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",  # Smooth convergence via cosine decay.
        seed=3407,
        report_to="none",
        output_dir=SAVE_DIR,
    ),
)

vessl_trainer = train_on_responses_only(
    vessl_trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

vessl_stats = vessl_trainer.train()

train_duration = time.time() - train_start
mins, secs = divmod(int(train_duration), 60)
print(f"\nTraining complete: {mins}m {secs:02d}s")

# Report loss trajectory.
if hasattr(vessl_stats, "log_history") and vessl_stats.log_history:
    losses = [h["loss"] for h in vessl_stats.log_history if "loss" in h]
    if losses:
        print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")


---
## 7. Test: ask the fine-tuned model about VESSL Cloud

Includes both in-distribution questions and held-out questions to
sanity-check generalisation.


In [ ]:
from transformers import TextStreamer

vessl_test_prompts = [
    # In-distribution: topics covered by the training data.
    "How do I pause a VESSL Cloud workspace to save cost?",
    "Which should I use, Cluster storage or Object storage?",
    # Out-of-distribution: generalisation checks.
    "Which GPU should I pick for LLM fine-tuning on VESSL Cloud?",
    "What's the cheapest way to run a workspace?",
]

for i, prompt in enumerate(vessl_test_prompts):
    print(f"\n{'='*60}")
    print(f"Test {i+1}: {prompt}")
    print(f"{'='*60}\n")

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    _ = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

print("\n" + "="*60)
print("Tests complete.")
print("="*60)


---
## 8. Save the adapter


In [ ]:
FINAL_PATH = f"{SAVE_DIR}/final"
os.makedirs(FINAL_PATH, exist_ok=True)

model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)

saved_files = os.listdir(FINAL_PATH)
total_size_mb = sum(
    os.path.getsize(os.path.join(FINAL_PATH, f))
    for f in saved_files
) / (1024 * 1024)

print(f"Saved VESSL-domain adapter.")
print(f"Location: {FINAL_PATH}")
print(f"Size: {total_size_mb:.0f} MB ({len(saved_files)} files)")
print("\nTeammates who mount the same Object volume can load this adapter directly.")


---
## Done

You now have a LoRA adapter trained on the VESSL QA dataset saved to
Object storage. Any teammate who mounts the same Object volume can load
the adapter with `PeftModel.from_pretrained`.

### Next steps
- Expand the dataset (100–500 samples) for more reliable domain facts.
- Try the vesslctl batch-job path (`../batch-job/`) for a reproducible,
  non-interactive run.
- **Stop your workspace** when done — Object storage persists across
  stops, so your adapter remains available.
